<a href="https://colab.research.google.com/github/w4bo/AA2627-unibo-mldm/blob/master/slides/lab-03-classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Classification: from ARFF data to defensible evaluation

## A hands-on lab with Bank, Census, and Nursery data

In this lab you will build classification workflows with the ARFF datasets in `slides/datasets`.

By the end, you should be able to:

- inspect ARFF data and identify features, targets, missing values, and identifiers;
- build leakage-safe preprocessing pipelines for numeric and categorical features;
- distinguish training, cross-validation, and test performance;
- compare a baseline, logistic regression, a decision tree, and k-nearest neighbours;
- interpret confusion matrices, precision, recall, F1, ROC AUC, and PR AUC;
- tune a model without using the test set;
- evaluate a selected model on a supplied external test set;
- extend the workflow from binary to multiclass classification.

**Suggested duration:** 2.5–3 hours. Complete the checkpoints before opening the suggested answers.

## 0. Setup and reproducibility

The notebook uses packages included in standard Google Colab runtimes. If a local environment is missing one, uncomment the installation command.

Run the notebook from top to bottom. All random operations use the same seed.

In [ ]:
# %pip install -q numpy pandas matplotlib scikit-learn

from pathlib import Path
import csv
import re
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    f1_score,
    precision_recall_curve,
    average_precision_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import (
    GridSearchCV,
    StratifiedKFold,
    cross_validate,
    train_test_split,
)
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier, plot_tree

RANDOM_STATE = 42
pd.set_option("display.max_columns", 50)
plt.style.use("seaborn-v0_8-whitegrid")

## 1. Load ARFF files without extra dependencies

ARFF stores a relation name, attribute declarations, and data rows. The helper below supports the dense ARFF files used in this course, including quoted categorical values and `?` missing-value markers.

`find_dataset` makes paths work whether the notebook starts in the repository root or in `slides/`.

In [ ]:
ATTRIBUTE_RE = re.compile(
    r"^@attribute\s+(?:'([^']+)'|\"([^\"]+)\"|(\S+))\s+(.+)$",
    flags=re.IGNORECASE,
)


def find_dataset(filename):
    candidates = [
        Path("datasets") / filename,
        Path("slides") / "datasets" / filename,
        Path.cwd() / "datasets" / filename,
        Path.cwd() / "slides" / "datasets" / filename,
    ]
    for path in candidates:
        if path.exists():
            return path.resolve()
    searched = "\n".join(f"- {p}" for p in candidates)
    raise FileNotFoundError(
        f"Could not find {filename}. Run from the repository root or slides/. "
        f"Searched:\n{searched}"
    )


def load_dense_arff(filename):
    path = find_dataset(filename)
    attributes = []
    data_lines = []
    in_data = False

    with path.open(encoding="utf-8-sig") as handle:
        for raw_line in handle:
            line = raw_line.strip()
            if not line or line.startswith("%"):
                continue
            if line.lower() == "@data":
                in_data = True
                continue
            if not in_data:
                match = ATTRIBUTE_RE.match(line)
                if match:
                    name = next(group for group in match.groups()[:3] if group is not None)
                    attributes.append((name, match.group(4).strip()))
            else:
                data_lines.append(line)

    rows = list(csv.reader(data_lines, quotechar="'", skipinitialspace=True))
    names = [name for name, _ in attributes]
    if not names or any(len(row) != len(names) for row in rows):
        raise ValueError(f"Unsupported or malformed ARFF file: {path}")

    frame = pd.DataFrame(rows, columns=names).replace("?", np.nan)
    for name, declared_type in attributes:
        if declared_type.lower() in {"numeric", "real", "integer"}:
            frame[name] = pd.to_numeric(frame[name], errors="coerce")
        else:
            values = frame[name].astype("string").str.strip()
            frame[name] = values.astype(object).where(values.notna(), np.nan)
    return frame

In [ ]:
bank = load_dense_arff("bank-data.arff")
census_train = load_dense_arff("CensusTraining.arff")
census_test = load_dense_arff("CensusTest.arff")
nursery = load_dense_arff("nurseryTrain.arff")

dataset_catalog = pd.DataFrame(
    {
        "rows": [len(bank), len(census_train), len(census_test), len(nursery)],
        "columns": [bank.shape[1], census_train.shape[1], census_test.shape[1], nursery.shape[1]],
        "target": ["pep", "class", "class", "class"],
    },
    index=["Bank", "Census train", "Census test", "Nursery"],
)
dataset_catalog

### Checkpoint 1 — understand the data contract

1. Which datasets have a separate, supplied test set?
2. Why should the Census test set remain untouched during model selection?
3. Which value represents missing data in an ARFF file?

<details>
<summary>Suggested answer</summary>

Only Census has an explicit training/test pair. Looking at its test results while choosing features or hyperparameters leaks test information into the model-selection process and makes the final estimate optimistic. ARFF conventionally uses `?` for a missing value.
</details>

# Part A — Bank Data: an end-to-end binary classification workflow

The task is to predict whether a customer buys a personal equity plan (`pep`). We will reserve one test set, compare candidate workflows by cross-validation on the training set, tune one model, and evaluate it once on the test set.

## 2. Audit the Bank dataset

A useful first audit checks shape, types, missingness, target balance, suspicious identifiers, and feature distributions. An identifier can let a model memorize rows without learning a reusable pattern.

In [ ]:
display(bank.head())

bank_audit = pd.DataFrame({
    "dtype": bank.dtypes.astype(str),
    "missing": bank.isna().sum(),
    "unique": bank.nunique(dropna=True),
})
display(bank_audit)
display(bank["pep"].value_counts().to_frame("count"))
display(bank["pep"].value_counts(normalize=True).rename("proportion").to_frame())

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for label, group in bank.groupby("pep"):
    axes[0].hist(group["income"], bins=15, alpha=0.55, label=label)
axes[0].set(title="Income by target", xlabel="Income", ylabel="Customers")
axes[0].legend(title="PEP")

pd.crosstab(bank["children"], bank["pep"], normalize="index").plot.bar(
    ax=axes[1], color=["#4C78A8", "#F58518"]
)
axes[1].set(title="Target proportion by children", ylabel="Proportion")
axes[1].legend(title="PEP")

pd.crosstab(bank["married"], bank["pep"], normalize="index").plot.bar(
    ax=axes[2], color=["#4C78A8", "#F58518"]
)
axes[2].set(title="Target proportion by marital status", ylabel="Proportion")
axes[2].legend(title="PEP")

plt.tight_layout()

### Exercise 1 — decide what enters the model (5 minutes)

Before running the next cell:

1. Should `id` be a feature? Why?
2. Which columns are numeric and which are categorical?
3. Is accuracy alone likely to be grossly misleading for this target?

<details>
<summary>Suggested answer</summary>

`id` should be removed: it uniquely names records and has no intended predictive meaning. `age`, `income`, and `children` are numeric; the remaining inputs are categorical. The target is not extremely imbalanced, so accuracy is informative, but balanced accuracy, F1, and the confusion matrix still expose different kinds of error.
</details>

## 3. Make a single stratified holdout split

The test set is placed “in a vault” until model selection is complete. Stratification preserves the target proportions in both partitions.

In [ ]:
X_bank = bank.drop(columns=["pep", "id"])
y_bank = bank["pep"]

X_train, X_test, y_train, y_test = train_test_split(
    X_bank,
    y_bank,
    test_size=0.25,
    random_state=RANDOM_STATE,
    stratify=y_bank,
)

split_balance = pd.concat(
    [
        y_train.value_counts(normalize=True).rename("train"),
        y_test.value_counts(normalize=True).rename("test"),
    ],
    axis=1,
)
print("Training shape:", X_train.shape)
print("Test shape:", X_test.shape)
display(split_balance)

## 4. Build leakage-safe preprocessing

A `Pipeline` learns preprocessing from each training fold only.

- Numeric columns: median imputation, then standardization.
- Categorical columns: most-frequent imputation, then one-hot encoding.
- Unknown categories are ignored so prediction does not fail on a new category.

Scaling matters for distance-based and linear models. It is unnecessary for a tree, but using a consistent transformer makes the comparison easy to audit.

In [ ]:
def make_preprocessor(X, scale_numeric=True):
    numeric = X.select_dtypes(include="number").columns.tolist()
    categorical = X.columns.difference(numeric).tolist()

    numeric_steps = [("imputer", SimpleImputer(strategy="median"))]
    if scale_numeric:
        numeric_steps.append(("scaler", StandardScaler()))

    return ColumnTransformer(
        transformers=[
            ("numeric", Pipeline(numeric_steps), numeric),
            (
                "categorical",
                Pipeline([
                    ("imputer", SimpleImputer(strategy="most_frequent")),
                    ("onehot", OneHotEncoder(handle_unknown="ignore")),
                ]),
                categorical,
            ),
        ]
    )


def classifier_pipeline(X, model, scale_numeric=True):
    return Pipeline([
        ("preprocess", make_preprocessor(X, scale_numeric=scale_numeric)),
        ("model", model),
    ])

## 5. Compare candidates with stratified cross-validation

Always include a simple baseline. `DummyClassifier` reveals whether a learned model improves on predicting the most common class.

We report the mean and standard deviation across five folds. The same folds are used for every candidate.

In [ ]:
models = {
    "Most-frequent baseline": classifier_pipeline(
        X_train, DummyClassifier(strategy="most_frequent"), scale_numeric=False
    ),
    "Logistic regression": classifier_pipeline(
        X_train, LogisticRegression(max_iter=2000, random_state=RANDOM_STATE)
    ),
    "Decision tree": classifier_pipeline(
        X_train,
        DecisionTreeClassifier(max_depth=5, min_samples_leaf=8, random_state=RANDOM_STATE),
        scale_numeric=False,
    ),
    "k-NN (k=9)": classifier_pipeline(X_train, KNeighborsClassifier(n_neighbors=9)),
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
scoring = {
    "accuracy": "accuracy",
    "balanced_accuracy": "balanced_accuracy",
    "f1": "f1_macro",
    "roc_auc": "roc_auc",
}

cv_rows = []
for name, workflow in models.items():
    scores = cross_validate(workflow, X_train, y_train, cv=cv, scoring=scoring)
    row = {"model": name}
    for metric in scoring:
        row[f"{metric}_mean"] = scores[f"test_{metric}"].mean()
        row[f"{metric}_sd"] = scores[f"test_{metric}"].std()
    cv_rows.append(row)

cv_results = pd.DataFrame(cv_rows).set_index("model").sort_values(
    "balanced_accuracy_mean", ascending=False
)
cv_results.round(3)

In [ ]:
ax = cv_results[["accuracy_mean", "balanced_accuracy_mean", "f1_mean", "roc_auc_mean"]].plot.bar(
    figsize=(11, 5), ylim=(0, 1), rot=20
)
ax.set(title="Bank: five-fold cross-validation", ylabel="Mean validation score", xlabel="")
ax.legend(loc="lower right")
plt.tight_layout()

### Checkpoint 2 — read cross-validation results

1. Does every learned model beat the baseline?
2. Are the model rankings identical for every metric?
3. Why is the standard deviation useful?
4. Why have we still not inspected test-set scores?

The standard deviation describes sensitivity to the particular validation fold. Small mean differences accompanied by large fold variation should not be over-interpreted.

## 6. Tune tree complexity using training data only

An unconstrained tree can memorize the training set. Grid search evaluates combinations of depth and minimum leaf size inside cross-validation. We optimize balanced accuracy and do not touch `X_test`.

In [ ]:
tree_workflow = classifier_pipeline(
    X_train,
    DecisionTreeClassifier(criterion="entropy", random_state=RANDOM_STATE),
    scale_numeric=False,
)

tree_grid = GridSearchCV(
    tree_workflow,
    param_grid={
        "model__max_depth": [2, 3, 4, 5, 7, None],
        "model__min_samples_leaf": [1, 5, 10, 20],
    },
    scoring="balanced_accuracy",
    cv=cv,
    n_jobs=1,
    return_train_score=True,
)
tree_grid.fit(X_train, y_train)

print("Best parameters:", tree_grid.best_params_)
print("Best mean CV balanced accuracy:", round(tree_grid.best_score_, 3))

In [ ]:
grid_results = pd.DataFrame(tree_grid.cv_results_)
grid_view = grid_results[[
    "param_model__max_depth",
    "param_model__min_samples_leaf",
    "mean_train_score",
    "mean_test_score",
    "std_test_score",
    "rank_test_score",
]].sort_values("rank_test_score")
grid_view.head(10).round(3)

### Exercise 2 — diagnose overfitting (10 minutes)

Use `grid_view` to find a configuration with a large train–validation gap.

1. Which setting overfits most strongly?
2. How do `max_depth` and `min_samples_leaf` control complexity?
3. Is the configuration with the highest training score necessarily the selected one?

<details>
<summary>Suggested answer</summary>

Deep trees with very small leaves usually have the largest gap. Restricting depth limits sequential splits; increasing minimum leaf size prevents rules based on very few rows. Grid search selects the highest mean validation score, not the highest training score.
</details>

## 7. Final Bank evaluation

Now evaluate the tuned workflow once on the held-out test set. The positive class is `YES`, so recall answers: “of customers who bought a PEP, how many did we find?”

In [ ]:
best_bank_model = tree_grid.best_estimator_
bank_pred = best_bank_model.predict(X_test)
bank_proba = best_bank_model.predict_proba(X_test)
positive_index = list(best_bank_model.classes_).index("YES")
bank_score = bank_proba[:, positive_index]

bank_test_metrics = pd.Series({
    "accuracy": accuracy_score(y_test, bank_pred),
    "balanced_accuracy": balanced_accuracy_score(y_test, bank_pred),
    "macro_f1": f1_score(y_test, bank_pred, average="macro"),
    "roc_auc": roc_auc_score((y_test == "YES").astype(int), bank_score),
    "average_precision": average_precision_score((y_test == "YES").astype(int), bank_score),
}, name="test_score")

display(bank_test_metrics.to_frame().round(3))
print(classification_report(y_test, bank_pred, digits=3))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

ConfusionMatrixDisplay.from_predictions(
    y_test, bank_pred, labels=["NO", "YES"], cmap="Blues", ax=axes[0], colorbar=False
)
axes[0].set_title("Confusion matrix")

y_test_binary = (y_test == "YES").astype(int)
fpr, tpr, _ = roc_curve(y_test_binary, bank_score)
axes[1].plot(fpr, tpr, label=f"AUC = {roc_auc_score(y_test_binary, bank_score):.3f}")
axes[1].plot([0, 1], [0, 1], "--", color="grey")
axes[1].set(xlabel="False-positive rate", ylabel="True-positive rate", title="ROC curve")
axes[1].legend()

precision, recall, _ = precision_recall_curve(y_test_binary, bank_score)
axes[2].plot(recall, precision, label=f"AP = {average_precision_score(y_test_binary, bank_score):.3f}")
axes[2].axhline(y_test_binary.mean(), linestyle="--", color="grey", label="Prevalence")
axes[2].set(xlabel="Recall", ylabel="Precision", title="Precision–recall curve")
axes[2].legend()

plt.tight_layout()

### Metric guide

| Metric | Main question | Common limitation |
|---|---|---|
| Accuracy | How often is the predicted label correct? | Can hide minority-class failure |
| Balanced accuracy | How good is average recall across classes? | Ignores probability quality |
| Precision | When the model predicts positive, how often is it right? | Falls when false positives rise |
| Recall | Of actual positives, how many are found? | Can be raised by predicting positive often |
| F1 | How strong is the precision–recall trade-off? | Hides which component is weak |
| ROC AUC | How well are positives ranked above negatives? | Can look optimistic with rare positives |
| PR AUC / AP | How precise is the ranking across recall levels? | Depends on positive prevalence |

The right metric depends on the cost of errors. A business policy needs those costs; the dataset alone cannot supply them.

## 8. Inspect the model, not just its score

Permutation importance measures how much test performance drops after shuffling one feature. It works on the original columns and can expose reliance on a small number of inputs. Importance is predictive, not causal.

In [ ]:
importance = permutation_importance(
    best_bank_model,
    X_test,
    y_test,
    scoring="balanced_accuracy",
    n_repeats=20,
    random_state=RANDOM_STATE,
    n_jobs=1,
)

importance_df = pd.DataFrame({
    "feature": X_test.columns,
    "importance_mean": importance.importances_mean,
    "importance_sd": importance.importances_std,
}).sort_values("importance_mean", ascending=False)

display(importance_df.round(3))
ax = importance_df.sort_values("importance_mean").plot.barh(
    x="feature", y="importance_mean", xerr="importance_sd", legend=False, figsize=(8, 5)
)
ax.set(title="Bank: permutation importance", xlabel="Decrease in balanced accuracy", ylabel="")
plt.tight_layout()

In [ ]:
fitted_tree = best_bank_model.named_steps["model"]
feature_names = best_bank_model.named_steps["preprocess"].get_feature_names_out()

plt.figure(figsize=(18, 8))
plot_tree(
    fitted_tree,
    feature_names=feature_names,
    class_names=fitted_tree.classes_,
    filled=True,
    rounded=True,
    max_depth=3,
    fontsize=8,
)
plt.title("First levels of the selected Bank decision tree")
plt.show()

### Exercise 3 — error analysis (10 minutes)

Inspect misclassified rows. Look for patterns, but do not treat a small test sample as proof.

In [ ]:
bank_errors = X_test.copy()
bank_errors["actual"] = y_test
bank_errors["predicted"] = bank_pred
bank_errors["p_yes"] = bank_score
bank_errors = bank_errors[bank_errors["actual"] != bank_errors["predicted"]]

print(f"Misclassified rows: {len(bank_errors)} / {len(X_test)}")
display(bank_errors.sort_values("p_yes").head(10))

# Part B — Census: supplied test data and imbalance

The Census task predicts whether annual income is `>50K`. Unlike Bank, this dataset supplies separate training and test ARFF files. We use training-only cross-validation for decisions and the supplied test set for final evaluation.

## 9. Audit train/test compatibility

Check that schemas match, inspect missingness, and compare class prevalence. `fnlwgt` is a survey weight rather than a personal characteristic; we exclude it here. `education` and `education-num` encode closely related information, so we remove the numeric duplicate for a clearer teaching model.

In [ ]:
assert census_train.columns.tolist() == census_test.columns.tolist()

TARGET = "class"
drop_columns = ["fnlwgt", "education-num"]

census_summary = pd.DataFrame({
    "train_missing": census_train.isna().sum(),
    "test_missing": census_test.isna().sum(),
    "train_unique": census_train.nunique(dropna=True),
    "test_unique": census_test.nunique(dropna=True),
})
display(census_summary)

prevalence = pd.concat(
    [
        census_train[TARGET].value_counts(normalize=True).rename("train"),
        census_test[TARGET].value_counts(normalize=True).rename("test"),
    ],
    axis=1,
)
display(prevalence)

In [ ]:
X_census_train = census_train.drop(columns=[TARGET] + drop_columns)
y_census_train = census_train[TARGET]
X_census_test = census_test.drop(columns=[TARGET] + drop_columns)
y_census_test = census_test[TARGET]

print("Training:", X_census_train.shape, "Test:", X_census_test.shape)

### Checkpoint 3 — distribution shift

1. Are the schemas identical?
2. Are missing values distributed identically?
3. Are target proportions identical?
4. If train and test distributions differ, what does that mean for interpretation?

A supplied test set can be harder or easier than the training sample. Differences do not automatically invalidate it, but they should be reported because they affect which population the score represents.

## 10. Select a Census model by cross-validation

The positive class is the less frequent `>50K` group. We compare a class-weighted logistic regression, a constrained decision tree, and a random forest. Class weighting asks the learner to penalize mistakes on the minority class more strongly; it does not change the evaluation data.

In [ ]:
census_models = {
    "Baseline": classifier_pipeline(
        X_census_train, DummyClassifier(strategy="most_frequent"), scale_numeric=False
    ),
    "Weighted logistic regression": classifier_pipeline(
        X_census_train,
        LogisticRegression(max_iter=3000, class_weight="balanced", random_state=RANDOM_STATE),
    ),
    "Weighted decision tree": classifier_pipeline(
        X_census_train,
        DecisionTreeClassifier(
            max_depth=6, min_samples_leaf=10, class_weight="balanced", random_state=RANDOM_STATE
        ),
        scale_numeric=False,
    ),
    "Weighted random forest": classifier_pipeline(
        X_census_train,
        RandomForestClassifier(
            n_estimators=300,
            min_samples_leaf=5,
            class_weight="balanced",
            random_state=RANDOM_STATE,
            n_jobs=1,
        ),
        scale_numeric=False,
    ),
}

census_cv_rows = []
for name, workflow in census_models.items():
    scores = cross_validate(workflow, X_census_train, y_census_train, cv=cv, scoring=scoring)
    census_cv_rows.append({
        "model": name,
        **{f"{metric}_mean": scores[f"test_{metric}"].mean() for metric in scoring},
        **{f"{metric}_sd": scores[f"test_{metric}"].std() for metric in scoring},
    })

census_cv_results = pd.DataFrame(census_cv_rows).set_index("model").sort_values(
    "balanced_accuracy_mean", ascending=False
)
census_cv_results.round(3)

## 11. Evaluate the selected Census model once

Choose the highest cross-validated balanced accuracy programmatically, refit it on all training rows, and evaluate it on the supplied test set.

In [ ]:
selected_name = census_cv_results["balanced_accuracy_mean"].idxmax()
selected_census_model = clone(census_models[selected_name]).fit(X_census_train, y_census_train)

census_pred = selected_census_model.predict(X_census_test)
census_proba = selected_census_model.predict_proba(X_census_test)
census_positive_index = list(selected_census_model.classes_).index(">50K")
census_score = census_proba[:, census_positive_index]
y_census_binary = (y_census_test == ">50K").astype(int)

print("Selected by CV:", selected_name)
display(pd.Series({
    "accuracy": accuracy_score(y_census_test, census_pred),
    "balanced_accuracy": balanced_accuracy_score(y_census_test, census_pred),
    "macro_f1": f1_score(y_census_test, census_pred, average="macro"),
    "roc_auc": roc_auc_score(y_census_binary, census_score),
    "average_precision": average_precision_score(y_census_binary, census_score),
}, name="supplied_test_score").to_frame().round(3))

print(classification_report(y_census_test, census_pred, digits=3))
ConfusionMatrixDisplay.from_predictions(
    y_census_test,
    census_pred,
    labels=["<=50K", ">50K"],
    cmap="Blues",
    colorbar=False,
)
plt.title("Census: supplied test set")
plt.show()

## 12. Slice-based error analysis

Aggregate metrics can hide uneven performance. The next cell reports accuracy and positive-class recall by sex. This is a descriptive diagnostic, **not** a fairness verdict: subgroup sizes, label quality, history, deployment context, and the costs of errors all matter.

In [ ]:
census_diagnostics = X_census_test[["sex"]].copy()
census_diagnostics["actual"] = y_census_test.to_numpy()
census_diagnostics["predicted"] = census_pred

slice_rows = []
for group, frame in census_diagnostics.groupby("sex"):
    actual_positive = frame["actual"] == ">50K"
    true_positive = actual_positive & (frame["predicted"] == ">50K")
    slice_rows.append({
        "sex": group,
        "n": len(frame),
        "positive_prevalence": actual_positive.mean(),
        "accuracy": (frame["actual"] == frame["predicted"]).mean(),
        "positive_recall": true_positive.sum() / max(actual_positive.sum(), 1),
    })

pd.DataFrame(slice_rows).set_index("sex").round(3)

### Exercise 4 — make a recommendation (15 minutes)

Write a short model card paragraph that includes:

1. the prediction target and intended evaluation population;
2. how the model was selected;
3. cross-validation and supplied-test performance;
4. the most important limitation you observed;
5. one additional check required before deployment.

Avoid saying the model is “good” based on accuracy alone.

# Part C — Nursery: multiclass extension

Nursery contains categorical features and five target classes. The same pipeline pattern works, but evaluation must account for multiple classes and strong imbalance.

In [ ]:
display(nursery.head())
display(nursery["class"].value_counts().to_frame("count"))

X_nursery = nursery.drop(columns="class")
y_nursery = nursery["class"]
X_nursery_train, X_nursery_test, y_nursery_train, y_nursery_test = train_test_split(
    X_nursery,
    y_nursery,
    test_size=0.25,
    random_state=RANDOM_STATE,
    stratify=y_nursery,
)

## 13. Multiclass challenge

Complete the workflow below.

1. Compare a baseline, decision tree, and logistic regression with five-fold CV.
2. Select using **macro F1**, which gives each class equal weight.
3. Fit the selected workflow and report a confusion matrix.
4. Identify the class with the lowest recall.

Why might accuracy and macro F1 rank models differently?

In [ ]:
# TODO: add at least two learned models.
nursery_models = {
    "Baseline": classifier_pipeline(
        X_nursery_train,
        DummyClassifier(strategy="most_frequent"),
        scale_numeric=False,
    ),
    # "Decision tree": classifier_pipeline(...),
    # "Logistic regression": classifier_pipeline(...),
}

# TODO: use cross_validate(..., scoring={"accuracy": "accuracy", "macro_f1": "f1_macro"})
# TODO: select, fit, predict, and display classification_report and a confusion matrix.

<details>
<summary>One possible Nursery solution</summary>

```python
nursery_models = {
    "Baseline": classifier_pipeline(
        X_nursery_train, DummyClassifier(strategy="most_frequent"), scale_numeric=False
    ),
    "Decision tree": classifier_pipeline(
        X_nursery_train,
        DecisionTreeClassifier(max_depth=12, min_samples_leaf=3, random_state=RANDOM_STATE),
        scale_numeric=False,
    ),
    "Logistic regression": classifier_pipeline(
        X_nursery_train,
        LogisticRegression(max_iter=3000, class_weight="balanced", random_state=RANDOM_STATE),
    ),
}

rows = []
for name, workflow in nursery_models.items():
    scores = cross_validate(
        workflow,
        X_nursery_train,
        y_nursery_train,
        cv=cv,
        scoring={"accuracy": "accuracy", "macro_f1": "f1_macro"},
    )
    rows.append({
        "model": name,
        "accuracy": scores["test_accuracy"].mean(),
        "macro_f1": scores["test_macro_f1"].mean(),
    })

nursery_results = pd.DataFrame(rows).set_index("model")
display(nursery_results.sort_values("macro_f1", ascending=False))

winner = nursery_results["macro_f1"].idxmax()
final_nursery = clone(nursery_models[winner]).fit(X_nursery_train, y_nursery_train)
nursery_pred = final_nursery.predict(X_nursery_test)
print(classification_report(y_nursery_test, nursery_pred, digits=3))
ConfusionMatrixDisplay.from_predictions(
    y_nursery_test, nursery_pred, xticks_rotation=45, cmap="Blues", colorbar=False
)
plt.title(f"Nursery test set — {winner}")
plt.tight_layout()
plt.show()
```

Accuracy is dominated by frequent classes. Macro F1 computes F1 per class and averages them equally, so failure on a rare class has a much larger effect.
</details>

## 14. What to submit

Submit the executed notebook plus a concise conclusion containing:

- the selected Bank model and why it was selected;
- its held-out confusion matrix and two metrics appropriate to the task;
- one example of overfitting from the grid-search results;
- the selected Census model and its supplied-test result;
- one limitation revealed by Census slice analysis;
- the completed Nursery multiclass challenge.

Do not report only the best number. Explain the evaluation design that makes the number credible.

## 15. Key takeaways

- Keep preprocessing inside the pipeline so each validation fold learns it independently.
- Keep a final test set out of model and hyperparameter selection.
- Compare against a baseline and report uncertainty across folds.
- Choose metrics from the error costs and class distribution, not convenience.
- Use confusion matrices and slice analysis to find failures hidden by averages.
- Treat feature importance as a diagnostic of predictive reliance, not a causal explanation.
- A supplied external test set measures generalization to that dataset's population, which may differ from training.